# Lekcija 11 - Protokol agent-prema-agentu (A2A)


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Što je A2A protokol?

**Agent-to-Agent (A2A) protokol** je otvoreni standard koji omogućuje AI agentima da komuniciraju,
otkriju jedni druge i surađuju — čak i kada su izgrađeni na različitim okvirima ili smješteni
kod različitih pružatelja usluga.

Ključni pojmovi:

- **Otkrivanje** – Agenti objavljuju *Karticu agenta* koja opisuje njihove sposobnosti, olakšavajući
  drugim agentima (ili orkestratorima) da pronađu pravog stručnjaka za zadatak.

- **Razmjena poruka** – Agenti razmjenjuju strukturirane poruke putem zajedničkog protokola, tako da
  zahtjev od jednog agenta može biti shvaćen i ispunjen od strane drugog bez obzira na njegovu internu
  implementaciju.

- **Životni ciklus zadatka** – A2A definira stanja kao što su *submitted*, *working*, *completed* i
  *failed*, dajući orkestratoru potpunu vidljivost o tome kako se delegirani zadatak odvija.

U ovoj lekciji simuliramo suradnju u A2A stilu povezivanjem triju specijaliziranih agenata za putovanja
u tijek rada gdje svaki agent doprinosi svojim stručnim znanjem i prosljeđuje rezultate sljedećem.


## Stvaranje specijaliziranih agenata za putovanja


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Suradnja više agenata putem tijeka rada

Povezujemo tri agenta u sekvencijalni tijek rada koji oponaša A2A prijenos poruka:

1. **CurrencyExchangeAgent** prima korisnički zahtjev i daje smjernice o valutama.
2. **ActivityPlannerAgent** prima obogaćeni kontekst i dodaje preporuke za aktivnosti.
3. **TravelManagerAgent** sintezira oba ulaza u konačni putni sažetak.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Razumijevanje A2A u produkciji

U produkcijskom okruženju A2A protokol otključava moćne međuslužbene scenarije:

| Mogućnost | Opis |
|---|---|
| **Interoperabilnost među okvirima** | Agent izgrađen u jednom okviru može delegirati zadatke agentu izgrađenom u bilo kojem drugom okviru kompatibilnom s A2A, omogućujući pravu interoperabilnost među organizacijama. |
| **Granice usluga** | Agenti mogu biti smješteni u zasebnim mikroservisima, regijama u oblaku ili čak različitim organizacijama, a i dalje besprijekorno surađivati. |
| **Dinamično otkrivanje** | Orkestrator može u vrijeme izvođenja upitati registar Agent Card kako bi pronašao najprikladnijeg specijalista za određeni podzadatak. |
| **Streaming & push notifications** | A2A podržava Server-Sent Events (SSE) za ažuriranja napretka u stvarnom vremenu i push obavijesti za dugotrajne zadatke. |

Radni tok koji smo gore izgradili je pojednostavljena verzija ovog obrasca koja se izvodi unutar istog procesa. U stvarnoj implementaciji svaki bi agent izlagao HTTP krajnju točku, objavio Agent Card i komunicirao putem A2A JSON-RPC protokola.


## Sažetak

U ovoj lekciji naučili ste:

1. **Što je A2A protokol** — otvoreni standard za otkrivanje agenata, razmjenu poruka i upravljanje zadacima.
2. **Kako stvoriti specijalizirane agente** — agent za razmjenu valuta, agent za planiranje aktivnosti i orkestrator za upravljanje putovanjima.
3. **Kako povezati agente u tijek rada** — korištenjem `WorkflowBuilder` za modeliranje sekvencijalnog prosljeđivanja poruka između agenata.
4. **Kako A2A radi u produkciji** — omogućavajući suradnju između različitih okvira i usluga s dinamičkim otkrivanjem i streaming ažuriranjima.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Odricanje odgovornosti:
Ovaj je dokument preveden pomoću AI prevoditeljske usluge Co-op Translator (https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na njegovom izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporučuje se profesionalni ljudski prijevod. Ne snosimo odgovornost za bilo kakve nesporazume ili pogrešna tumačenja koja proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
